# 03 - ML tanpa Spark (scikit-learn)
Logistic Regression, Random Forest, Gradient Boosting. Split temporal.

In [ ]:
import os, sys, json, time
import pandas as pd

BASE_DIR = os.path.abspath(os.environ.get("BDA_BASE_DIR", ".."))
print("BASE_DIR aktif:", BASE_DIR)
assert os.path.isdir(os.path.join(BASE_DIR, "src")), (
    f"BASE_DIR salah: {BASE_DIR} tidak punya folder src/. "
    "Set os.environ['BDA_BASE_DIR'] ke path repo yang benar sebelum run cell ini."
)

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
FIG_DIR = os.path.join(BASE_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

sys.path.insert(0, os.path.join(BASE_DIR, "src"))
import features as ft

df = pd.read_parquet(os.path.join(PROCESSED_DIR, "earthquake_features.parquet"))
print(f"Loaded: {df.shape}")


In [ ]:
# Feature engineering tambahan dulu (energy, shallow_flag, event_count 7d/30d, days_since_last_event)
# -- label butuh zone_id + shallow_flag + event_count_30d yang dihasilkan di sini
df = ft.add_ml_features(df)
df[ft.FEATURE_COLUMNS].describe()


In [ ]:
# Label risk_label = proxy risiko ZONA (histori event_count_30d + shallow_ratio zona),
# BUKAN dari mag/sig event itu sendiri -- supaya model tidak menebak ulang inputnya sendiri.
# mag, sig, event_count_30d, shallow_flag sengaja dikeluarkan dari FEATURE_COLUMNS karena
# jadi komponen langsung risk_score (lihat src/features.py untuk detail alasan).
df = ft.add_risk_label(df)
print("Threshold label:", df.attrs["risk_label_thresholds"])
print(df["risk_label"].value_counts(normalize=True).round(3))


In [ ]:
# Split TEMPORAL: train <=2022, test 2023-2026 (bukan random -- cegah leakage time-series)
train_df, test_df = ft.temporal_split(df, train_until_year=2022)
print(f"train: {len(train_df)} ({train_df['year'].min()}-{train_df['year'].max()})")
print(f"test:  {len(test_df)} ({test_df['year'].min()}-{test_df['year'].max()})")


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), ft.FEATURE_COLUMNS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ft.CATEGORICAL_COLUMNS),
])

X_train = train_df[ft.FEATURE_COLUMNS + ft.CATEGORICAL_COLUMNS]
y_train = train_df[ft.TARGET_COLUMN]
X_test = test_df[ft.FEATURE_COLUMNS + ft.CATEGORICAL_COLUMNS]
y_test = test_df[ft.TARGET_COLUMN]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "logistic_regression": LogisticRegression(max_iter=1000, multi_class="multinomial"),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

fitted = {}
timing = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    timing[name] = round(time.time() - t0, 3)
    fitted[name] = pipe
    print(f"{name}: fit {timing[name]}s")


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    confusion_matrix, classification_report,
)
from sklearn.preprocessing import label_binarize
import numpy as np

classes = sorted(y_train.unique())
metrics_summary = []

for name, pipe in fitted.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    y_test_bin = label_binarize(y_test, classes=classes)
    auc = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")

    metrics_summary.append(dict(
        model=name, accuracy=round(acc, 4), precision_macro=round(precision, 4),
        recall_macro=round(recall, 4), f1_macro=round(f1, 4), roc_auc_ovr=round(auc, 4),
        fit_time_s=timing[name],
    ))
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, zero_division=0))

metrics_df = pd.DataFrame(metrics_summary)
metrics_df


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pipe) in zip(axes, fitted.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=classes)
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes, ax=ax, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "03_confusion_matrices.png"), dpi=150)
plt.show()


In [ ]:
# Feature importance (Random Forest & Gradient Boosting)
cat_names = list(fitted["random_forest"].named_steps["prep"].transformers_[1][1].get_feature_names_out(ft.CATEGORICAL_COLUMNS))
all_feature_names = ft.FEATURE_COLUMNS + cat_names

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name in zip(axes, ["random_forest", "gradient_boosting"]):
    importances = fitted[name].named_steps["clf"].feature_importances_
    imp_series = pd.Series(importances, index=all_feature_names).sort_values(ascending=True)
    imp_series.plot(kind="barh", ax=ax)
    ax.set_title(f"Feature Importance - {name}")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "03_feature_importance.png"), dpi=150)
plt.show()


In [ ]:
# Simpan metrik untuk dibandingkan dengan Spark MLlib (notebook 04)
metrics_df.to_json(os.path.join(PROCESSED_DIR, "metrics_sklearn.json"), orient="records")
print("Tersimpan: metrics_sklearn.json")
metrics_df


In [ ]:
# Push hasil (metrics json, figures) ke GitHub supaya bisa dicek tanpa Drive
import sync
sync.push_results(BASE_DIR, message="Update hasil 03_ml_sklearn")
